# Week 4 Activity: Delay, Filters, Reverb

Complete this activity as part of your participation grade. Pending length of the lecture, you will have time in class to work. Everything you need to complete this activity can be found in this week's (or a previous week's) lecture code. For this activity, you may want to consult your Audio Tech I notes (or the Computer Music Tutorial)

## Delay
1) Create a function that will modify a passed signal by adding a delay of $m$ milliseconds to the signal.

In [32]:
import numpy as np
from scipy.io.wavfile import read
from scipy.signal import sawtooth, square, butter, filtfilt, butter, lfilter
import matplotlib.pyplot as plt
from IPython.display import Audio, Image

In [33]:

def delay (x, fs, m):
    data = x/abs(x).max()
    copy = data.copy()
    pad = np.zeros(int(fs/1000*m))
    delay = np.concatenate([pad,copy])
    return delay


2) Modify the function such that you can optionally scale the amplitude of the delayed signal 

In [34]:
def delay (x, fs, m, a = 1.0): 
    data = x/abs(x).max()
    copy = data.copy()
    pad = np.zeros(int(fs/1000*m))
    delay = np.concatenate([pad,copy])
    delay *= a
    return delay

3) Modify the function so that the user can specify the number of delays to add to the original signal (and optionally, scale each subsequent delay amplitude by a factor of $1/n$

In [35]:
def delay(x, fs, m, numdelays=1, scale=False):
    data = x / np.abs(x).max()
    delay = data.copy()
    delay_samples = int(fs * m / 1000)

    for n in range(1, numdelays+1):
        pad = np.zeros(delay_samples*n)
        y = np.concatenate([pad, data])
        if scale:
            y *= 1/n 
        if len(y) > len(delay):
            delay = np.pad(delay, (0,len(y)-len(delay)))
        delay[:len(y)] +=y
    
    return delay


## Filters
1. Create a function that will apply an feedforward comb filter by computing the delay length based off a given resonant frequency in Hz.

In [36]:
def feedforwardcomb(x, fs, f, g=1.0):
    M = int(round(fs/f))
    response = np.zeros(M + 1)
    response[0]= 1
    response[M]= g

    out = np.convolve(x, response, mode = 'same')

    return out


2. Create a function that will apply an feedback comb filter by computing the delay length based off a given resonant frequency in Hz.

In [37]:
import numpy as np

def feedbackcomb(x, f, f0, g=0.5):
    M = int(round(f / f0))
    out = np.zeros_like(x)
    
    for n in range(len(x)):
        out[n] = x[n]
        if n-M >= 0:
            out[n]+=g*out[n-M]


    return out


4. Use your comb filter functions on a wave from the audio folder. Try applying different resonant frequencies and delay lengths. How are the filter results different?

In [38]:
fs, x = read("../audio/80spopDrums.wav")
forward = feedforwardcomb(x, fs, 400)
back = feedbackcomb(x, fs, 400)
Audio(back, rate = fs) 

3. Create a function that will apply a butterworth filter to a signal with filter type options 'highpass', 'lowpass', 'bandpass', and 'bandstop'.

In [39]:
def butterworth_filter(x, fs, cutoff, order=4, btype='lowpass'):
    nyq = fs/2
    if btype in ['bandpass','bandstop']:
        Wn = [cutoff[0]/nyq, cutoff[1]/nyq]
    else:
        Wn = cutoff/nyq
    b, a = butter(order, Wn, btype=btype)
    y = lfilter(b,a,x)
    return y

## Reverb/Convolution

1. Create a function that will apply a simple moving average filter by convolving the filter kernel and an incoming signal.

In [40]:
import numpy as np

def movingavg(x, N):
    h = np.ones(N)/N
    out = np.convolve(x, h, mode = 'same')

    return out


2. Apply your filter to a noise signal. What is the effect? What happens if you increase or decrease the kernel size?

In [41]:
noise = np.random.randn(44100)
kernel = 200 
x = movingavg(noise, kernel)
Audio(x, rate=fs)

3. Create a function that applies convolution reverb to an input signal given an impulse response (this can be default loaded from the audio files). Use np.convolve to create this function.

In [42]:
def convreverb(x, response, norm=True):
    out = np.convolve(x,response)
    if norm:
        out= out/np.max(np.abs(out))
    return out

4. Create another function that applies convolution reverb to an input signal given an impulse response, but this time do not use np.convolve. You should write the convolution from scratch.

    Challenge yourself to create the most efficient function and time your implementation against np.convolve. 
    While developing and testing your function, do not use real audio files. Start with short signals (e.g., impulses, noise, or short sinusoids). Using long signals with loop-based implementations will result in extremely slow run times.

    **Hint:** in a similar manner to how you may have designed your delay function, recall the functions `numpy.zeros` and `numpy.roll` along with how to manipulate multidimensional numpy arrays (e.g., scalar product, summing columns with `vstack`, etc.) If you plan to try `numpy.roll` DO NOT use it in a loop for convolution with a real audio file! (You'll kill your memory), instead consider the `map` function. You may also want to check out the following function which is similar to numpy.roll but more efficient for this task: `scipy.linalg.circulant`.

    You may wish to visit [here](https://numpy.org/doc/stable/user/basics.broadcasting.html) for review of broadcasting (i.e., form some calculation across index value I and column value C) in numpy

In [43]:
def convolution_reverb_manual(x, ir):
    N = len(x)
    M = len(ir)
    x_padded = np.pad(x,(M-1, M-1))
    X = np.lib.stride_tricks.sliding_window_view(x_padded,M)
    out = X @ ir

    return out
